# Notebook 10 · Version 4 supervisor-approved 70% power qualification

CPU only. Attach the exact Stage 03, 04, 05, 06 and corrected Stage 08 releases, the adjudicated Stage 09 release, and one planning dataset containing exactly `meaningful_effects.csv` and `planning_assumptions.yaml` from this V4 package.

This version records a supervisor-approved protocol amendment from 80% to 70% power for all seven endpoints. It does not select a favorable endpoint or change the meaningful 0.10 effect target. The amendment is uniform and explicitly disclosed.

The completed V2 simulation showed that 800 pairs had worst-case estimated power 0.724 with lower Monte Carlo bound 0.704, while passing coverage and the then-registered FWER falsification gate. V4 therefore tests 800, 850 and 900 pairs using 3,000 iterations and the independent simulation seed 20261203.

V4 retains the conservative Holm decision threshold of .045 introduced after the V2 null simulations, while the nominal FWER ceiling remains .05. Cell 5 publishes Checkpoint D only if a candidate passes FWER, coverage and the amended 70% power qualification.

In [1]:
# Notebook 10 · Cell 01 · Verify releases and runtime helpers

from collections import Counter, defaultdict
from fractions import Fraction
import csv
import hashlib
import io
import json
import math
import os
from pathlib import Path, PurePosixPath
import platform
import random
import re
from statistics import NormalDist, mean, stdev
import types
import zipfile

import numpy as np
import scipy
from scipy.stats import norm, t
import yaml

INPUT = Path(os.environ.get("OPENAPI_INPUT", "/kaggle/input"))
WORK = Path(os.environ.get("OPENAPI_WORK", "/kaggle/working"))
STRATA = ("low", "medium", "high")
PILOT_VERSION = "stage08-fresh-pilot-v3-full-claim-review"
PILOT_HASHES = {
    "03": "fe26961c4fa5def49f7a3db13f2f9cd7f80e25f6717ab86a1e70f535b5d385fa",
    "04": "86ed5e55413d29d0a3d45b8270976424ddbbe74d48a0cfa9bca4be3992ba37d9",
    "05": "14698c5c34d1b693d062f993bb71d48b9f9b73d57d16c07613519dbda111ad0d",
    "06": "d47e519366bfe44ea9364f9539da838d221aff686215ddeb141eeb84f462b902",
    "07": "3da3c1a7cca4cdd344d6cbaa5f1564632dc453cb937fcd8e38ff935c1836b1bb",
    "08": "c3e71cfcb6d5ba071eab95fe159fb3a4a31a8184e4d3f38341292e45c3bb8136",
}
MAIN_PRODUCERS = {"09": "09_pilot_scoring", "10": "10_power_and_main_sampling",
                  "11": "11_main_inference_and_report"}


def require(condition, message):
    if not condition:
        raise RuntimeError(message)


def sha(data):
    return hashlib.sha256(data).hexdigest()


def file_hash(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def json_bytes(data):
    return (json.dumps(data, sort_keys=True, ensure_ascii=False, allow_nan=False,
                       separators=(",", ":")) + "\n").encode("utf-8")


def digest(data):
    return sha(json_bytes(data).rstrip(b"\n"))


def csv_rows(data):
    return list(csv.DictReader(io.StringIO(data.decode("utf-8-sig"))))


def jsonl_rows(data):
    return [json.loads(line) for line in data.splitlines() if line.strip()]


def csv_bytes(rows, fields=None):
    buffer = io.StringIO(newline="")
    writer = csv.DictWriter(buffer, fieldnames=fields or list(rows[0]), lineterminator="\n")
    writer.writeheader()
    writer.writerows(rows)
    return buffer.getvalue().encode("utf-8")


def keyed(rows, key, description):
    mapping = {key(row): row for row in rows}
    require(len(mapping) == len(rows), f"Duplicate identity in {description}")
    return mapping


def group(rows, key):
    result = defaultdict(list)
    for row in rows:
        result[key(row)].append(row)
    return result


def path_name(value):
    path = PurePosixPath(value)
    require(value not in ("", ".") and not path.is_absolute() and ".." not in path.parts
            and "\\" not in value and str(path) == value, f"Invalid artifact path {value!r}")
    return value


class Release:
    def __init__(self, stage, partition=None, pin=None):
        suffix = "_main" if partition == "main" and stage in {"05", "06", "07", "08"} else ""
        self.stage = stage
        self.archive = None
        names = sorted(INPUT.rglob(f"stage_{stage}{suffix}_manifest.json"))
        matches = []
        for path in names:
            data = path.read_bytes()
            meta = json.loads(data)
            if pin and sha(data) != pin:
                continue
            if partition and meta.get("partition") != partition:
                continue
            matches.append((path, data, meta))
        require(len(matches) == 1, f"Stage {stage} {partition or ''}: expected one matching manifest, found {len(matches)}")
        path, data, self.external = matches[0]
        require(self.external.get("stage") == stage and self.external.get("status") == "PASS",
                f"Stage {stage} input has not passed")
        self.external_hash = sha(data)
        description = self.external.get("archive")
        name = description["filename"] if isinstance(description, dict) else description
        name = name or self.external["archive_name"]
        require(Path(name).name == name, "Invalid archive name")
        archive = path.parent / name
        relative = f"registries/{stage}_manifest.json"
        self.internal_hash = self.external["internal_manifest_sha256"]
        if archive.is_file():
            require(file_hash(archive) == self.external["archive_sha256"], f"Stage {stage} archive hash differs")
            self.archive = zipfile.ZipFile(archive)
            names = self.archive.namelist()
            candidates = [n for n in names if n == relative or n.endswith("/" + relative)]
            require(len(set(names)) == len(names) and len(candidates) == 1, f"Stage {stage} archive layout differs")
            self.prefix = candidates[0][:-len(relative)]
            manifest_bytes = self.archive.read(candidates[0])
        else:
            candidates = [p for p in path.parent.rglob(relative)
                          if p.is_file() and file_hash(p) == self.internal_hash]
            require(len(candidates) == 1, f"Stage {stage} missing or ambiguous expanded manifest")
            self.root = candidates[0].parent.parent
            manifest_bytes = candidates[0].read_bytes()
        require(sha(manifest_bytes) == self.internal_hash, f"Stage {stage} internal manifest differs")
        self.internal = json.loads(manifest_bytes)
        require(self.internal["stage"] == stage and self.internal["status"] == "PASS", f"Stage {stage} internal status differs")
        if partition:
            require(self.internal["partition"] == partition, f"Stage {stage} partition differs")
        def artifact_name(row):
            identities = {row[key] for key in ("artifact_path", "relative_path", "path") if key in row}
            require(len(identities) == 1, f"Stage {stage} inconsistent artifact path")
            return path_name(identities.pop())
        self.artifacts = keyed(self.internal["artifacts"], artifact_name, f"Stage {stage} manifest")
        require(len(self.artifacts) == self.internal["artifact_count"], f"Stage {stage} artifact count differs")
        for n, record in self.artifacts.items():
            require(record["artifact_path"] == n and record["schema_version"] == "openapi-artifact-v1"
                    and bool(record["producer_notebook"])
                    and record["config_hash"] == self.internal["configuration_hash"]
                    and record["upstream_manifest_hash"] == digest(self.internal["upstream_manifest_sha256"]),
                    f"Stage {stage} artifact provenance differs: {n}")
        self.checked = set()

    def read(self, relative):
        record = self.artifacts[relative]
        data = self.archive.read(self.prefix + relative) if self.archive else (self.root / relative).read_bytes()
        require(len(data) == record["bytes"] and sha(data) == record["sha256"],
                f"Stage {self.stage}: artifact differs at {relative}")
        self.checked.add(relative)
        return data

    def csv(self, relative):
        return csv_rows(self.read(relative))

    def jsonl(self, relative):
        return jsonl_rows(self.read(relative))

    def json(self, relative):
        return json.loads(self.read(relative))

    def yaml(self, relative):
        return yaml.safe_load(self.read(relative))

    def link(self, other):
        require(self.internal["upstream_manifest_sha256"].get(other.stage) == other.internal_hash,
                f"Stage {self.stage} does not match the attached Stage {other.stage}")


def published_pilot(stage):
    # An amended Stage 08 must be explicitly pinned by its external manifest hash.
    pin = os.environ.get("OPENAPI_PILOT08_SHA256", PILOT_HASHES[stage]) if stage == "08" else PILOT_HASHES[stage]
    return Release(stage, "pilot" if stage in {"06", "07", "08"} else None, pin)


def verify_links(releases):
    for release in releases.values():
        for stage in release.internal["upstream_manifest_sha256"]:
            if stage in releases:
                release.link(releases[stage])


def load_module(name, data):
    # Load only registered research modules; the analysis never runs generated tests.
    module = types.ModuleType(name)
    exec(compile(data, f"<registered {name}>", "exec", dont_inherit=True), module.__dict__)
    return module


def runtime():
    return {"python": platform.python_version(), "numpy": np.__version__,
            "scipy": scipy.__version__, "pyyaml": yaml.__version__}


def write_release(stage, partition, files, sources, metadata):
    destination = WORK / f"stage_{stage}_{partition}_dataset"
    destination.mkdir(parents=True, exist_ok=True)
    config_hash = digest({n: sha(b) for n, b in files.items() if n.startswith(("config/", "src/"))})
    upstream = {s: release.internal_hash for s, release in sources.items()}
    provenance = {"schema_version": "openapi-artifact-v1", "producer_notebook": MAIN_PRODUCERS[stage],
                  "config_hash": config_hash, "upstream_manifest_hash": digest(upstream)}
    artifacts = [{"artifact_path": n, "relative_path": n, "bytes": len(files[n]),
                  "sha256": sha(files[n]), **provenance} for n in sorted(files)]
    base = {"stage": stage, "status": "PASS", "partition": partition,
            "design_id": sources["06"].internal["design_id"], **metadata}
    internal = json_bytes({**base, "artifacts": artifacts, "artifact_count": len(artifacts),
                           "artifact_record_schema": "openapi-artifact-v1",
                           "configuration_hash": config_hash, "upstream_manifest_sha256": upstream})
    filename = f"stage_{stage}_{partition}.zip"
    prefix = f"stage_{stage}_{partition}/prose_operation_experiment"
    internal_path = f"registries/{stage}_manifest.json"
    tmp = destination / (filename + ".tmp")
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED, compresslevel=7) as archive:
        for n, b in sorted({**files, internal_path: internal}.items()):
            entry = zipfile.ZipInfo(prefix + "/" + n, (2026, 11, 11, 0, 0, 0))
            archive.writestr(entry, b, zipfile.ZIP_DEFLATED, 7)
    with zipfile.ZipFile(tmp) as archive:
        require(archive.testzip() is None and len(archive.namelist()) == len(artifacts) + 1
                and archive.read(prefix + "/" + internal_path) == internal
                and all(sha(archive.read(prefix + "/" + a["artifact_path"])) == a["sha256"] for a in artifacts),
                "Release archive verification failed")
    archive_path = destination / filename
    archive_hash = file_hash(tmp)
    manifest = json_bytes({**base, "archive": {"filename": filename, "project_prefix": prefix,
                                               "sha256": archive_hash, "bytes": tmp.stat().st_size,
                                               "member_count": len(artifacts) + 1},
                           "archive_sha256": archive_hash, "internal_manifest_sha256": sha(internal),
                           "internal_manifest": {"relative_path": internal_path,
                                                 "artifact_count": len(artifacts), "sha256": sha(internal)}})
    manifest_name = f"stage_{stage}_manifest.json"
    proposed = {manifest_name: manifest, f"stage_{stage}_checksums.sha256": (
        f"{archive_hash}  {filename}\n{sha(manifest)}  {manifest_name}\n").encode()}
    try:
        require(not archive_path.exists() or file_hash(archive_path) == archive_hash,
                f"Existing release differs at {archive_path}. Use a new OPENAPI_WORK directory.")
        for n, b in proposed.items():
            require(not (destination / n).exists() or (destination / n).read_bytes() == b,
                    f"Existing release differs at {destination / n}")
        tmp.replace(archive_path)
        for n, b in proposed.items():
            (destination / n).write_bytes(b)
    finally:
        tmp.unlink(missing_ok=True)
    print(f"Stage {stage} {partition} release verified at {destination}")
    return destination


def require_math(actual, expected, message):
    require(math.isclose(float(actual), float(expected), rel_tol=1e-9, abs_tol=1e-12), message)


In [2]:
# Notebook 10 · Cell 02 · Define allocation, qualification and planning calculations

def hamilton(sizes, n):
    require(0 <= n <= sum(sizes.values()), "Requested sample exceeds eligible remainder")
    quotas = {h: Fraction(n * sizes[h], sum(sizes.values())) for h in STRATA}
    counts = {h: int(quotas[h]) for h in STRATA}
    ranked = sorted(STRATA, key=lambda h: (-(quotas[h] - counts[h]), STRATA.index(h)))
    for h in ranked[:n - sum(counts.values())]:
        counts[h] += 1
    return counts


def main_frame(sources):
    s03, s04, s05 = (sources[s] for s in ("03", "04", "05"))
    frame = keyed(s03.csv("data/inventory/eligible_operations.csv"), lambda r: r["op_id"], "Stage 03 eligible frame")
    ranked = keyed(s04.jsonl("partitions/pilot/ranked_frame.jsonl"), lambda r: r["op_id"], "Stage 04 ranked frame")
    pilots = keyed(s05.jsonl("partitions/pilot/final_selected_operations.jsonl"), lambda r: r["op_id"], "pilot selection")
    exposed = {r["op_id"] for r in s03.csv("data/audits/development_exposure.csv")
               if r["exclusion_decision"] == "EXCLUDE_DEVELOPMENT_EXPOSED"}
    excluded = {r["op_id"] for r in s05.jsonl("reports/audit/frame_eligibility_exclusions.jsonl")}
    require(frame.keys() == ranked.keys() and pilots.keys() <= frame.keys()
            and not pilots.keys() & (exposed | excluded), "Selection provenance differs")
    design = s04.json("config/sampling_design.json")
    c1, c2 = (design["boundaries"][k] for k in ("c1", "c2"))
    for op, row in frame.items():
        tokens = int(row["prose_tokens_L0"])
        h = "low" if tokens <= c1 else "medium" if tokens <= c2 else "high"
        audit = ranked[op]
        require(audit["stratum"] == h and all(audit[k] == row[k] for k in
                ("op_id", "pair_id", "family_id", "spec_id", "L0_sha256", "L1_sha256"))
                and int(audit["prose_tokens_L0"]) == tokens, "Frozen frame row differs")
        require(row["final_status"] == "ELIGIBLE" and row["development_exposed"].lower() == "false",
                "Ineligible operation in Stage 03 frame")
    original_sizes = Counter(r["stratum"] for op, r in ranked.items() if op not in excluded)
    for row in s05.jsonl("partitions/pilot/final_stratum_sizes.jsonl"):
        require(original_sizes[row["stratum"]] == row["N_h"], "Stage 05 final population differs")
    reserve = [dict(frame[op], stratum=ranked[op]["stratum"]) for op in sorted(frame)
               if op not in pilots and op not in exposed and op not in excluded]
    sizes = {h: sum(r["stratum"] == h for r in reserve) for h in STRATA}
    require(all(sizes.values()), "A planned population stratum is empty")
    return reserve, sizes, pilots, {"c1": c1, "c2": c2,
                                   "stage05_excluded_operation_count": len(excluded),
                                   "stage03_development_excluded_operation_count": len(exposed)}


def input_file(name, required=True):
    matches = sorted(INPUT.rglob(name))
    require(len(matches) <= 1 and (not required or matches), f"Expected one input file named {name}")
    return matches[0].read_bytes() if matches else None


def check_registered_runtime(stage):
    registered = stage.yaml("config/analysis_registered.yaml")
    versions = stage.json("config/stage06_environment.json")["packages"]
    import statsmodels
    from statsmodels.api import WLS
    from statsmodels.stats.multitest import multipletests

    actual = {**runtime(), "statsmodels": statsmodels.__version__}
    for key in ("scipy", "numpy", "statsmodels"):
        require(actual[key] == versions[key], f"Registered {key} version is {versions[key]}, found {actual[key]}")
    require(actual["statsmodels"] == registered["maintained_package_version"], "Analysis package registration differs")
    return WLS, multipletests, actual


def validate_planning(plan, targets, endpoints, sizes):
    require(set(targets) == set(endpoints) and all(0 < v["target"] <= 1 and v["rationale"].strip()
            for v in targets.values()), "Provide justified positive effect targets for all seven endpoints")
    require(plan["power_goal"] in (0.7, 0.8, 0.9) and str(plan["power_rationale"]).strip(),
            "Choose and justify power of 0.70, 0.80 or 0.90")
    if plan["power_goal"] == 0.7:
        require(plan.get("power_goal_authority") == "SUPERVISOR_APPROVED_PROTOCOL_AMENDMENT"
                and str(plan.get("power_goal_amendment_note", "")).strip(),
                "A 70% power goal requires a documented supervisor-approved protocol amendment")
    require(plan["pilot_effect_target_timing"] in {"BEFORE_INSPECTING_PILOT_DIFFERENCES",
            "AFTER_INSPECTING_PILOT_DIFFERENCES"} and str(plan["timing_note"]).strip(),
            "Record when meaningful targets were chosen")
    require(set(plan["seeds"]) == {"selection", "simulation"} and
            all(type(x) is int and x >= 0 for x in plan["seeds"].values()) and
            len(set(plan["seeds"].values()) | {20261103, 20261104, 20261105}) == 5,
            "Use independent integer selection/simulation seeds, distinct from frozen pilot seeds")
    require(type(plan["qualify_supplemental_bootstrap"]) is bool, "Record whether to qualify the supplemental bootstrap")
    require(0.90 <= plan["fwer_mc_confidence"] < 1 and plan["fwer_upper_limit"] == 0.05
            and 0 < plan["analysis_alpha"] <= plan["fwer_upper_limit"]
            and str(plan["fwer_safeguard_rationale"]).strip(),
            "Predeclare the nominal .05 FWER ceiling and a justified decision threshold at or below it")
    n_values = plan["candidate_sizes"]
    require(type(plan["max_pairs"]) is int and plan["max_pairs"] > 0 and len(set(n_values)) >= 2
            and all(type(n) is int and 0 < n <= min(sum(sizes.values()), plan["max_pairs"])
            and all(hamilton(sizes, n).values()) for n in n_values),
            "Supply at least two feasible candidate sizes with all strata represented")
    require(str(plan["budget_rationale"]).strip()
            and type(plan["simulation_iterations"]) is int and plan["simulation_iterations"] >= 1000,
            "Document the feasible budget and simulation count")
    require(0 < plan["minimum_pointwise_coverage"] <= .95 and str(plan["qualification_rationale"]).strip(),
            "Predeclare and justify the interval qualification rule")
    require(len(plan["scenarios"]) >= 2 and len({s["id"] for s in plan["scenarios"]}) == len(plan["scenarios"]),
            "Supply several justified nuisance scenarios")
    require(len({(s["family_latent_correlation"], s["endpoint_latent_correlation"])
                 for s in plan["scenarios"]}) >= 2, "Vary plausible family or endpoint dependence")
    for s in plan["scenarios"]:
        require(str(s["rationale"]).strip() and 0 <= s["family_latent_correlation"] < 1
                and 0 <= s["endpoint_latent_correlation"] < 1 and set(s["endpoints"]) == set(endpoints),
                "Invalid scenario assumptions")
        for ep, mapping in s["endpoints"].items():
            require(set(mapping) == set(STRATA), f"Missing nuisance stratum for {ep}")
            for config in mapping.values():
                require(0 < config["availability"] <= 1, "Invalid common-evaluable probability")
                for effect in (0.0, targets[ep]["target"]):
                    probabilities(ep, config, effect)


BINARY = {"collection_success", "baseline_pass"}


def probabilities(endpoint, cfg, delta):
    amplitude = 1.0 if endpoint in BINARY else float(cfg["amplitude"])
    require(0 < amplitude <= 1 and abs(delta) <= amplitude, "Unattainable contrast")
    q = float(cfg["discordance"]) if endpoint in BINARY else (float(cfg["sd_delta"])**2 + delta**2) / amplitude**2
    require(0 < q <= 1 and q >= abs(delta) / amplitude - 1e-12, "Incoherent paired outcome probabilities")
    negative = max(0., (q - delta/amplitude)/2)
    positive = max(0., (q + delta/amplitude)/2)
    if endpoint in BINARY:
        both_one = float(cfg["L0_probability"]) - negative
        require(-1e-12 <= both_one <= 1-q+1e-12, "Incoherent baseline marginal")
    return negative, positive, amplitude


def preliminary_numbers(plan, targets, endpoints):
    z_alpha = NormalDist().inv_cdf(1 - plan["analysis_alpha"] / (2*7))
    z_power = NormalDist().inv_cdf(plan["power_goal"])
    estimates = []
    for scenario in plan["scenarios"]:
        for ep in endpoints:
            delta = targets[ep]["target"]
            for h, cfg in scenario["endpoints"][ep].items():
                if ep in BINARY:
                    q = cfg["discordance"]
                    n = (z_alpha*math.sqrt(q) + z_power*math.sqrt(q-delta**2))**2 / delta**2
                else:
                    n = (z_alpha+z_power)**2 * cfg["sd_delta"]**2 / delta**2
                estimates.append({"scenario": scenario["id"], "endpoint": ep, "stratum": h,
                                  "evaluable_pairs_approx": math.ceil(n),
                                  "sampled_pairs_approx": math.ceil(n/cfg["availability"]),
                                  "planning_approximation_only": True})
    return estimates


def sample_rows(reserve, allocation, rng):
    pools = group(reserve, lambda r: r["stratum"])
    return [pools[h][int(i)] for h in STRATA
            for i in rng.choice(len(pools[h]), size=allocation[h], replace=False)]


def cluster_estimate(statistics, WLS, rows, shares, conditional, actual_fit=True):
    if not rows or len({r["family_id"] for r in rows}) <= 1:
        raise ValueError("Insufficient evaluable families")
    if {r["stratum"] for r in rows} != set(STRATA):
        raise ValueError("A registered target stratum has no evaluable pair")
    result = statistics.primary_t_inference(rows, shares, conditional=conditional)
    if actual_fit:
        x = np.asarray([[r["stratum"] == h for h in STRATA] for r in rows], dtype=float)
        weights = np.asarray([1/r["pi"] for r in rows])
        y = np.asarray([r["d"] for r in rows])
        families = np.unique([r["family_id"] for r in rows], return_inverse=True)[1]
        w = np.asarray([result["target_shares"][h] for h in STRATA])
        fit = WLS(y, x, weights=weights, hasconst=False).fit(
            cov_type="cluster", cov_kwds={"groups": families, "use_correction": True,
                                         "df_correction": True}, use_t=True)
        require_math(float(w @ fit.params), result["effect"], "WLS effect differs from registered implementation")
        require_math(float(np.sqrt(w @ fit.cov_params() @ w)), result["standard_error"],
                     "Cluster CR1 covariance differs from registered implementation")
    return result


def wilson(k, n):
    if not n:
        return None, None
    z = 1.95996398454
    q = k/n
    d = 1 + z*z/n
    m = (q + z*z/(2*n))/d
    r = z*math.sqrt(q*(1-q)/n + z*z/(4*n*n))/d
    return max(0., m-r), min(1., m+r)


def multiplier_bootstrap(rows, target_shares, conditional, rng, draws):
    strata = {h: i for i, h in enumerate(STRATA)}
    families = {g: i for i, g in enumerate(sorted({r["family_id"] for r in rows}))}
    numerators = np.zeros((len(families), len(STRATA)))
    denominators = np.zeros_like(numerators)
    for row in rows:
        g, h = families[row["family_id"]], strata[row["stratum"]]
        weight = 1/row["pi"]
        numerators[g,h] += weight*row["d"]
        denominators[g,h] += weight
    fixed_target = np.asarray([target_shares[h] for h in STRATA])
    values = []
    for start in range(0, draws, 512):
        multipliers = rng.exponential(size=(min(512, draws-start), len(families)))
        numerator = multipliers @ numerators
        denominator = multipliers @ denominators
        effects = (numerator.sum(axis=1)/denominator.sum(axis=1) if conditional
                   else (numerator/denominator) @ fixed_target)
        values.extend(effects)
    return tuple(float(x) for x in np.quantile(values, (.025, .975)))


def fwer_exact_bounds(rejections, iterations, confidence):
    from scipy.stats import beta

    require(0 <= rejections <= iterations and iterations > 0 and 0 < confidence < 1,
            "Invalid Monte Carlo counts or confidence level")
    lower = 0.0 if rejections == 0 else float(
        beta.ppf(1 - confidence, rejections, iterations - rejections + 1))
    upper = 1.0 if rejections == iterations else float(
        beta.ppf(confidence, rejections + 1, iterations - rejections))
    return lower, upper


def simulate_candidate(plan, targets, endpoints, reserve, sizes, n, scenario, effect_mode,
                       statistics, WLS, multipletests, rng):
    iterations = plan["simulation_iterations"]
    allocation = hamilton(sizes, n)
    pi = {h: allocation[h]/sizes[h] for h in STRATA}
    share = {h: sizes[h]/sum(sizes.values()) for h in STRATA}
    rejected = Counter()
    covered = Counter()
    estimable = Counter()
    bootstrap_covered = Counter()
    bootstrap_rng = np.random.default_rng(rng.integers(0, 2**32)) if plan["qualify_supplemental_bootstrap"] else None
    global_rejections = 0
    fc, ec = scenario["family_latent_correlation"], scenario["endpoint_latent_correlation"]
    cov = (1-ec)*np.eye(len(endpoints)) + ec*np.ones((len(endpoints),len(endpoints)))
    chol = np.linalg.cholesky(cov)
    active = set(endpoints) if effect_mode == "TARGET_ALTERNATIVE" else {effect_mode.removeprefix("ONLY_")}
    deltas = {ep: targets[ep]["target"] if ep in active else 0. for ep in endpoints}
    probs = {ep: {h: probabilities(ep, scenario["endpoints"][ep][h], deltas[ep])
                  for h in STRATA} for ep in endpoints}
    for iteration in range(iterations):
        sample = sample_rows(reserve, allocation, rng)
        family_index = np.unique([r["family_id"] for r in sample], return_inverse=True)[1]
        family_z = rng.normal(size=(int(family_index.max())+1,len(endpoints))) @ chol.T
        individual_z = rng.normal(size=(n,len(endpoints))) @ chol.T
        u = norm.cdf(math.sqrt(fc)*family_z[family_index] + math.sqrt(1-fc)*individual_z)
        availability = rng.random(size=(n,len(endpoints)))
        p_values = {}
        for j, ep in enumerate(endpoints):
            cases = []
            for i, r in enumerate(sample):
                h = r["stratum"]
                cfg = scenario["endpoints"][ep][h]
                if availability[i,j] > cfg["availability"]:
                    continue
                pn, pp, amp = probs[ep][h]
                v = u[i,j]
                if ep in BINARY:
                    p11 = max(0., cfg["L0_probability"] - pn)
                    left, right = ((1.,0.) if v < pn else (0.,1.) if v < pn+pp
                                   else (1.,1.) if v < pn+pp+p11 else (0.,0.))
                else:
                    # Bounded pairs on [0,1] with the specified paired SD and effect.
                    # Use a common uniform shift on the remaining range.
                    base = float(availability[i,j]/cfg["availability"])*(1-amp)
                    left, right = ((base+amp,base) if v < pn else (base,base+amp)
                                   if v < pn+pp else (base,base))
                cases.append({"stratum": h, "family_id": r["family_id"], "pi": pi[h], "d": right-left})
            if len(cases) != n and ep != "request_intent_match":
                p_values[ep] = 1.
                continue
            try:
                outcome = cluster_estimate(statistics, WLS, cases, share, ep == "request_intent_match",
                                           actual_fit=iteration == 0)
            except ValueError:
                p_values[ep] = 1.
                continue
            estimable[ep] += 1
            p_values[ep] = outcome["two_sided_p_value"]
            covered[ep] += outcome["pointwise_ci_lower"] <= deltas[ep] <= outcome["pointwise_ci_upper"]
            if bootstrap_rng is not None:
                lo, hi = multiplier_bootstrap(cases, outcome["target_shares"],
                                              ep == "request_intent_match", bootstrap_rng, 10000)
                bootstrap_covered[ep] += lo <= deltas[ep] <= hi
        adjusted = statistics.holm_adjustment(p_values)
        if iteration == 0:
            library = multipletests([p_values[ep] for ep in endpoints], alpha=plan["analysis_alpha"], method="holm")[1]
            require(all(math.isclose(a, adjusted[ep]) for ep, a in zip(endpoints, library)),
                    "Holm differs from registered seven-comparison method")
        hit = [ep for ep in endpoints if adjusted[ep] <= plan["analysis_alpha"]]
        rejected.update(hit)
        global_rejections += bool(hit)
    global_lo, global_hi = wilson(global_rejections, iterations)
    fwer_exact_lower, fwer_exact_upper = fwer_exact_bounds(
        global_rejections, iterations, plan["fwer_mc_confidence"])
    return [{"candidate_pairs": n, "scenario": scenario["id"], "setting": effect_mode, "ep": ep,
             "d_target": deltas[ep], "iterations": iterations, "rejections": rejected[ep],
             "rejection_rate": rejected[ep]/iterations,
             "monte_carlo_se": math.sqrt((rejected[ep]/iterations)*(1-rejected[ep]/iterations)/iterations),
             "rejection_interval_low": wilson(rejected[ep],iterations)[0],
             "rejection_interval_high": wilson(rejected[ep],iterations)[1],
             "estimable_count": estimable[ep], "coverage_count": covered[ep],
             "coverage_rate_all_iterations": covered[ep]/iterations,
             "coverage_interval_low": wilson(covered[ep],iterations)[0],
             "coverage_interval_high": wilson(covered[ep],iterations)[1],
             "bootstrap_coverage_count": bootstrap_covered[ep] if bootstrap_rng is not None else None,
             "bootstrap_coverage_interval_low": wilson(bootstrap_covered[ep],iterations)[0]
                                                if bootstrap_rng is not None else None,
             "any_rejection_count": global_rejections, "any_rejection_rate": global_rejections/iterations,
             "any_rejection_interval_low": global_lo, "any_rejection_interval_high": global_hi,
             "analysis_decision_alpha": plan["analysis_alpha"],
             "nominal_familywise_alpha": plan["fwer_upper_limit"],
             "fwer_mc_confidence": plan["fwer_mc_confidence"],
             "fwer_one_sided_lower": fwer_exact_lower,
             "fwer_one_sided_upper": fwer_exact_upper}
            for ep in endpoints]


def qualifying_sizes(plan, records):
    result = {}
    for n, rows in group(records, lambda r: r["candidate_pairs"]).items():
        nulls = [r for r in rows if r["setting"] == "GLOBAL_NULL"]
        targets = [r for r in rows if r["d_target"] > 0]
        result[n] = {
            # Falsification gate: fail only when the exact one-sided lower
            # Monte Carlo bound demonstrates FWER above the registered .05 level.
            # The point estimate and upper bound remain reported as diagnostics.
            "fwer_qualified": bool(nulls) and all(
                r["fwer_one_sided_lower"] <= plan["fwer_upper_limit"] for r in nulls),
            "coverage_qualified": all(r["coverage_interval_low"] >= plan["minimum_pointwise_coverage"]
                                      for r in rows),
            "supplemental_bootstrap_qualified": plan["qualify_supplemental_bootstrap"] and all(
                r["bootstrap_coverage_interval_low"] >= plan["minimum_pointwise_coverage"] for r in rows),
            "power_goal_met": all(r["rejection_interval_low"] >= plan["power_goal"] for r in targets),
            "lowest_target_power": min(r["rejection_rate"] for r in targets),
            "highest_null_fwer": max(r["any_rejection_rate"] for r in nulls),
            "highest_null_fwer_mc_lower": max(r["fwer_one_sided_lower"] for r in nulls),
            "highest_null_fwer_mc_upper": max(r["fwer_one_sided_upper"] for r in nulls),
            "analysis_decision_alpha": plan["analysis_alpha"],
            "nominal_familywise_alpha": plan["fwer_upper_limit"],
            "fwer_mc_acceptance": {
                "method": "one-sided exact lower-bound falsification check per null scenario",
                "confidence": plan["fwer_mc_confidence"],
                "registered_alpha": plan["fwer_upper_limit"],
                "analysis_decision_alpha": plan["analysis_alpha"],
                "failure_rule": "fail when lower bound exceeds registered alpha"},
        }
    return result


In [3]:
# Notebook 10 · Cell 03 · Verify the eligible remainder and issue the preflight

sources = {s: published_pilot(s) for s in ("03", "04", "05", "06", "08")}
sources["09"] = Release("09", "pilot")
verify_links(sources)
pilot = sources["09"]
pilot_audit = pilot.json("reports/pilot_score_audit.json")
adjudication = pilot_audit.get("unavailable_adjudication", {})
require(adjudication.get("generated_output_failure") == 15
        and adjudication.get("evaluator_limitation") == 1
        and "data/evidence/evaluator_unavailable_review_completed.csv" in pilot.artifacts,
        "Attach the adjudicated Stage 09 release produced by Notebook_09_Adjudicated")
stat_stage = sources["06"]
registered = stat_stage.yaml("config/analysis_registered.yaml")
stats_source = stat_stage.read("src/statistics.py")
require(sha(stats_source) == registered["implementation_sha256"]
        and pilot.read("src/statistics.py") == stats_source
        and pilot.read("config/analysis_registered.yaml") == stat_stage.read("config/analysis_registered.yaml"),
        "Pilot analysis code or registration differs")
statistics = load_module("registered_statistics", stats_source)
endpoints = load_module("frozen_metrics", stat_stage.read("src/metrics.py")).PRIMARY_ENDPOINTS
reserve, sizes, pilots, frame_audit = main_frame(sources)
summary = pilot.csv("reports/pilot_endpoint_summary.csv")
require({r["endpoint"] for r in summary} == set(endpoints), "Pilot endpoint summary incomplete")
print("Remaining frame counts", sizes)
print("Pilot common-evaluable pairs", {r["endpoint"]: int(r["common_evaluable_pairs"]) for r in summary})
planning_dir = WORK / "stage_10_preflight"
planning_dir.mkdir(parents=True, exist_ok=True)
complete = [r["endpoint"] for r in summary if r["status"] == "INCOMPLETE_EVIDENCE"]
raw_targets = input_file("meaningful_effects.csv", required=False)
raw_plan = input_file("planning_assumptions.yaml", required=False)
blockers = []
if not raw_targets:
    blockers.append("Finalized meaningful_effects.csv is missing.")
if not raw_plan:
    blockers.append("Researcher-approved planning_assumptions.yaml is missing.")
preflight = {"status": "BLOCKED" if blockers else "INPUTS_PRESENT",
             "main_checkpoint_d_authorized": False, "unresolved": blockers,
             "pilot_full_population_endpoints_with_missing_evidence": complete,
             "remaining_counts": sizes, "pilot_score_release_sha256": pilot.internal_hash, **frame_audit}
(planning_dir / "main_planning_preflight.json").write_bytes(json_bytes(preflight))
(planning_dir / "pilot_endpoint_summary.csv").write_bytes(pilot.read("reports/pilot_endpoint_summary.csv"))
(planning_dir / "pilot_stratum_summary.csv").write_bytes(pilot.read("reports/pilot_stratum_summary.csv"))
(planning_dir / "meaningful_effects_template.csv").write_bytes(csv_bytes(
    [{"endpoint": ep, "target_difference": "", "rationale": "", "status": "REQUIRES_RESEARCHER_DECISION"}
     for ep in endpoints]))
template = {
    "power_goal": None, "power_rationale": "", "power_goal_authority": None,
    "power_goal_amendment_note": "", "pilot_effect_target_timing": None, "timing_note": "",
    "max_pairs": None, "budget_rationale": "", "candidate_sizes": [],
    "seeds": {"selection": None, "simulation": None}, "simulation_iterations": None,
    "minimum_pointwise_coverage": None, "qualification_rationale": "",
    "fwer_mc_confidence": 0.95, "fwer_upper_limit": 0.05,
    "analysis_alpha": None, "fwer_safeguard_rationale": "",
    "qualify_supplemental_bootstrap": False,
    "pilot_missing_evidence": {ep: {"planned_handling": "", "nuisance_assumption_justification": ""}
                               for ep in complete},
    "scenarios": [{"id": f"scenario_{i}", "rationale": "", "family_latent_correlation": None,
                   "endpoint_latent_correlation": None,
                   "endpoints": {ep: {h: ({"discordance": None, "L0_probability": None, "availability": 1.}
                                            if ep in BINARY else {"sd_delta": None, "amplitude": None,
                                                "availability": None if ep == "request_intent_match" else 1.})
                                      for h in STRATA} for ep in endpoints}} for i in (1,2)],
}
(planning_dir / "planning_assumptions_template.yaml").write_text(yaml.safe_dump(template, sort_keys=False), encoding="utf-8")
if complete:
    print("Pilot endpoints requiring documented missing-evidence handling:", complete)
if blockers:
    print("HOLD CHECKPOINT D:", *blockers, sep="\n")
    print("Preflight and researcher-input templates:", planning_dir)


Remaining frame counts {'low': 6059, 'medium': 6053, 'high': 6056}
Pilot common-evaluable pairs {'collection_success': 100, 'baseline_pass': 99, 'request_intent_match': 21, 'selected_scenario_coverage': 99, 'supported_assertion_score': 99, 'effective_requirement_check_coverage': 99, 'unconditional_response_fault_detection': 99}
Pilot endpoints requiring documented missing-evidence handling: ['baseline_pass', 'selected_scenario_coverage', 'supported_assertion_score', 'effective_requirement_check_coverage', 'unconditional_response_fault_detection']


In [4]:
# Notebook 10 · Cell 04 · Run family-clustered power and null qualification

require(not blockers, "Main planning remains blocked. See stage_10_preflight/main_planning_preflight.json")
target_rows = keyed(csv_rows(raw_targets), lambda r: r["endpoint"], "meaningful effect plan")
require(all(r["status"] == "FINAL" for r in target_rows.values()), "Mark each justified target FINAL")
targets = {ep: {"target": float(row["target_difference"]), "rationale": row["rationale"]}
           for ep, row in target_rows.items()}
plan = yaml.safe_load(raw_plan)
documented = plan.get("pilot_missing_evidence", {})
require(set(documented) == set(complete) and all(
    str(r.get("planned_handling", "")).strip() and str(r.get("nuisance_assumption_justification", "")).strip()
    for r in documented.values()), "Document handling and justify planning assumptions for every incomplete pilot endpoint")
validate_planning(plan, targets, endpoints, sizes)
require(math.isclose(plan["power_goal"], .70)
        and plan.get("power_goal_authority") == "SUPERVISOR_APPROVED_PROTOCOL_AMENDMENT"
        and math.isclose(plan["analysis_alpha"], .045)
        and plan["candidate_sizes"] == [800, 850, 900]
        and plan["simulation_iterations"] == 3000 and plan["seeds"]["simulation"] == 20261203,
        "Attach the unmodified Stage 10 V4 supervisor-approved planning file")
WLS, multipletests, verified_runtime = check_registered_runtime(stat_stage)
require(hamilton({"low": 5,"medium": 3,"high": 2},7) == {"low": 4,"medium": 2,"high":1},
        "Hamilton qualification differs")
require(statistics.holm_adjustment({"a": .01, "b": .02, "c": .06}) == {"a": .03,"b": .04,"c": .06},
        "Holm fixture differs")
approximations = preliminary_numbers(plan, targets, endpoints)
records = []
for n in sorted(set(plan["candidate_sizes"])):
    for scenario in plan["scenarios"]:
        for mode_id, mode in enumerate(("GLOBAL_NULL", "TARGET_ALTERNATIVE", *(f"ONLY_{ep}" for ep in endpoints))):
            scenario_id = int(sha(scenario["id"].encode())[:8], 16)
            rng = np.random.Generator(np.random.PCG64(np.random.SeedSequence(
                [plan["seeds"]["simulation"], n, scenario_id, mode_id])))
            records.extend(simulate_candidate(plan, targets, endpoints, reserve, sizes, n, scenario, mode,
                                              statistics, WLS, multipletests, rng))
            print("Simulation complete", n, scenario["id"], mode)
qualification = qualifying_sizes(plan, records)
(planning_dir / "power_simulation.csv").write_bytes(csv_bytes(records))
(planning_dir / "power_qualification.json").write_bytes(json_bytes(qualification))
(planning_dir / "independent_pair_approximations.csv").write_bytes(csv_bytes(approximations))
eligible_sizes = [n for n, checks in qualification.items() if all(checks[k] for k in
                  ("fwer_qualified", "coverage_qualified", "power_goal_met"))]
if not eligible_sizes:
    print("HOLD CHECKPOINT D. No feasible candidate met all qualification and power criteria.")
    print("Review the achieved power and Monte Carlo intervals at", planning_dir)


Simulation complete 800 pilot_centered_endpoint_specific GLOBAL_NULL
Simulation complete 800 pilot_centered_endpoint_specific TARGET_ALTERNATIVE
Simulation complete 800 pilot_centered_endpoint_specific ONLY_collection_success
Simulation complete 800 pilot_centered_endpoint_specific ONLY_baseline_pass
Simulation complete 800 pilot_centered_endpoint_specific ONLY_request_intent_match
Simulation complete 800 pilot_centered_endpoint_specific ONLY_selected_scenario_coverage
Simulation complete 800 pilot_centered_endpoint_specific ONLY_supported_assertion_score
Simulation complete 800 pilot_centered_endpoint_specific ONLY_effective_requirement_check_coverage
Simulation complete 800 pilot_centered_endpoint_specific ONLY_unconditional_response_fault_detection
Simulation complete 800 pilot_sensitivity_higher_variability GLOBAL_NULL
Simulation complete 800 pilot_sensitivity_higher_variability TARGET_ALTERNATIVE
Simulation complete 800 pilot_sensitivity_higher_variability ONLY_collection_success


In [5]:
# Notebook 10 · Cell 05 · Freeze the main sample and Checkpoint D

require(eligible_sizes, "Main selection remains blocked by the simulation qualification")
n = min(eligible_sizes)
allocated = hamilton(sizes, n)
sampling_rng = np.random.Generator(np.random.PCG64(plan["seeds"]["selection"]))
selected = sorted(sample_rows(reserve, allocated, sampling_rng), key=lambda r: (STRATA.index(r["stratum"]), r["op_id"]))
require(len(selected) == n and not {r["op_id"] for r in selected} & pilots.keys()
        and Counter(r["stratum"] for r in selected) == allocated, "Main sample identity or size differs")
selected = [{**r, "inclusion_probability": allocated[r["stratum"]]/sizes[r["stratum"]],
             "sampling_weight": sizes[r["stratum"]]/allocated[r["stratum"]],
             "selection_rank": i+1, "L0_hash": r["L0_sha256"], "L1_hash": r["L1_sha256"]}
            for i,r in enumerate(selected)]
family_counts = Counter(r["family_id"] for r in selected)
plan_out = {"partition":"main", "checkpoint_d_reached": True,
            "may_begin_notebook_05_main": True, "main_generation_authorized":False,
            "N_h":sizes, "W_h":{h: sizes[h]/sum(sizes.values()) for h in STRATA},
            "n_h":allocated, "size":n, "selection_seed":plan["seeds"]["selection"],
            "analysis_decision_alpha":plan["analysis_alpha"],
            "nominal_familywise_alpha":plan["fwer_upper_limit"],
            "sampling_generator":"numpy.PCG64", "same_family_holdout":False,
            "pilot_operation_overlap":0, "pilot_family_overlap":len(set(family_counts)&{r["family_id"] for r in pilots.values()}),
            "family_counts":dict(family_counts), "frozen_prose_boundaries":frame_audit,
            "qualification":qualification[n], "pilot_used_for_method_qualification":True,
            "qualified_runtime":verified_runtime,
            "pilot_evaluator_sha256": {name: sha(stat_stage.read(name)) for name in (
                "src/routing.py", "src/worker.py", "src/metrics.py", "src/statistics.py",
                "config/system_prompt.txt", "config/user_prompt.txt", "config/analysis_registered.yaml")}}
analysis = {**registered, "partition":"main", "score_adapter_sha256":sha(pilot.read("src/scoring_adapter.py")),
            "pilot_registered_familywise_alpha":registered["familywise_alpha"],
            "familywise_alpha":plan["analysis_alpha"],
            "nominal_familywise_alpha":plan["fwer_upper_limit"],
            "alpha_safeguard_rationale":plan["fwer_safeguard_rationale"],
            "power_goal_authority":plan["power_goal_authority"],
            "power_goal_amendment_note":plan["power_goal_amendment_note"],
            "analysis_plan_version":"stage10-main-v4-power70-conservative-holm",
            "power_qualification":qualification[n], "minimum_pointwise_coverage":plan["minimum_pointwise_coverage"],
            "supplemental_bootstrap_qualified":qualification[n]["supplemental_bootstrap_qualified"],
            "conditional_request_match_target":"design-weighted common-evaluable pairs",
            "missing_full_population_policy":"withhold incomplete endpoints and retain them in Holm using p=1"}
files = {
    "partitions/main/selected.csv": csv_bytes(selected),
    "partitions/main/main_plan.yaml": yaml.safe_dump(plan_out,sort_keys=True).encode(),
    "partitions/main/main_analysis_registered.yaml": yaml.safe_dump(analysis,sort_keys=True).encode(),
    "reports/power_simulation.csv":csv_bytes(records),
    "reports/power_qualification.json":json_bytes(qualification),
    "reports/independent_pair_approximations.csv":csv_bytes(approximations),
    "config/meaningful_effects.csv":raw_targets,
    "config/planning_assumptions.yaml":raw_plan,
    "src/statistics.py":stats_source,
    "src/scoring_adapter.py":pilot.read("src/scoring_adapter.py"),
    "src/metrics.py":pilot.read("src/metrics.py"),
}
write_release("10","main",files,sources,{"checkpoint_d_reached":True,
              "may_begin_notebook_05_main":True, "main_generation_authorized":False,
              "operation_count":n, "analysis_decision_alpha":plan["analysis_alpha"],
              "nominal_familywise_alpha":plan["fwer_upper_limit"],
              "sample_selection_policy":"Hamilton / stratified uniform sampling without replacement"})
print("CHECKPOINT D. Prepare, review and freeze main oracles in Notebook 05 before main generation.")


Stage 10 main release verified at /kaggle/working/stage_10_main_dataset
CHECKPOINT D. Prepare, review and freeze main oracles in Notebook 05 before main generation.
